# KUL CV GA2 — Ensemble Notebook (Kaggle)

将多个已训练模型的概率进行加权融合，生成最终分类提交文件。

## 工作流程

1. 在验证集上搜索最优的 **per-class 概率权重**（逐类别找最高 mAP 权重组合）
2. 对测试集应用相同权重 → 生成加权概率
3. 用 per-class 最优阈值（F1 最大化）二值化 → 生成 1500 行提交 CSV

## 使用前提

**方案 A（推荐）：本 notebook 内同时完成训练+融合**
- 直接在本 notebook 中训练所有模型，然后执行 ensemble 单元格。

**方案 B：加载已有概率 CSV**
- 将各模型的 `test_probabilities_*.csv` 和 `val_probabilities_*.csv` 上传为 Kaggle Dataset，
  然后修改下方 `PROB_SOURCES` 字典中的路径。

**方案 C：加载已有 checkpoint**
- 将各模型的 `best_model.pth` 上传为 Kaggle Dataset，
  修改下方 `CHECKPOINT_SOURCES`，本 notebook 自动推理并融合。

本 notebook 默认采用 **方案 C**（推荐，适合 Kaggle 多 notebook 流程）。

In [ ]:
from pathlib import Path
import json, random
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, f1_score
from tqdm.auto import tqdm

print('torch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    !nvidia-smi

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 核心配置 — 根据实际情况修改
# ══════════════════════════════════════════════════════════════════════════════

DATA_DIR    = Path('/kaggle/input/kul-computer-vision-ga-2-2026')
ENSEMBLE_NAME = 'ensemble_kaggle'   # 输出目录名
OUT = Path('/kaggle/working') / ENSEMBLE_NAME

# ── 方案 C：从 checkpoint 推理（修改 slug 和路径匹配你的 Kaggle Dataset 输入）──
# 格式：{实验名: (backbone名称, feat_dim, checkpoint路径, img_size)}
# 如果某个模型没有 checkpoint，直接删除该行。
CHECKPOINT_SOURCES = {
    'convnext_small_320':    ('convnext_small',  768,  Path('/kaggle/input/convnext-small-checkpoint/best_model.pth'),  320),
    'convnext_base_320':     ('convnext_base',   1024, Path('/kaggle/input/convnext-base-checkpoint/best_model.pth'),   320),
    'efficientnet_v2_s_320': ('efficientnet_v2_s', 1280, Path('/kaggle/input/effv2s-checkpoint/best_model.pth'),        320),
}

# ── 方案 B：直接加载已有概率 CSV（若使用此方案，把方案 C 设为 {}）──
# 格式：{实验名: (val_prob_csv路径, test_prob_csv路径)}
PROB_SOURCES = {}
# 示例：
# PROB_SOURCES = {
#     'convnext_small_320':    (Path('/kaggle/input/probs/val_probabilities_convnext_small_320.csv'),
#                               Path('/kaggle/input/probs/test_probabilities_convnext_small_320.csv')),
# }

# ── Ensemble 超参数 ────────────────────────────────────────────────────────────
GRID_STEP   = 0.25   # 权重搜索步长（0.25 速度快；0.1 更精细但慢）
VAL_SPLIT   = 0.2
SEED        = 42
NUM_WORKERS = 2
EVAL_BS     = 32
USE_AMP     = True
WEAK_CLASSES = ['bottle', 'diningtable', 'pottedplant', 'sheep', 'sofa']

LABELS = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor',
]

MET, PRED, SUB, FIG = OUT/'metrics', OUT/'predictions', OUT/'submissions', OUT/'figures'
for d in (MET, PRED, SUB, FIG): d.mkdir(parents=True, exist_ok=True)
print('Output:', OUT)

In [ ]:
# ── 数据集 & 工具 ──────────────────────────────────────────────────────────────
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

_MEAN, _STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def val_tfm(sz=320):
    return T.Compose([T.Resize((sz,sz)), T.ToTensor(), T.Normalize(_MEAN,_STD)])

class VOCDataset(Dataset):
    def __init__(self, df, data_dir, split='train', transform=None):
        self.df, self.data_dir, self.split = df, Path(data_dir), split
        self.transform = transform or val_tfm()
        self.has_labels = all(c in df.columns for c in LABELS)
        self.indices = list(df.index)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        img = Image.fromarray(np.load(self.data_dir/self.split/'img'/f'{self.split}_{idx}.npy'))
        img = self.transform(img)
        if self.has_labels:
            return img, torch.FloatTensor(self.df.loc[idx, LABELS].values.astype(float))
        return img, idx

def rle_encode(arr):
    pixels = np.concatenate([[0], arr.flatten(), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

pin = device.type == 'cuda'

In [ ]:
# ── 模型工厂（方案 C 使用）─────────────────────────────────────────────────────
_BACKBONES = {
    'convnext_tiny': {
        'factory': lambda: models.convnext_tiny(weights=None),
        'features': lambda m: nn.Sequential(m.features, m.avgpool),
        'feat_dim': 768,
    },
    'convnext_small': {
        'factory': lambda: models.convnext_small(weights=None),
        'features': lambda m: nn.Sequential(m.features, m.avgpool),
        'feat_dim': 768,
    },
    'convnext_base': {
        'factory': lambda: models.convnext_base(weights=None),
        'features': lambda m: nn.Sequential(m.features, m.avgpool),
        'feat_dim': 1024,
    },
    'convnext_large': {
        'factory': lambda: models.convnext_large(weights=None),
        'features': lambda m: nn.Sequential(m.features, m.avgpool),
        'feat_dim': 1536,
    },
    'efficientnet_b3': {
        'factory': lambda: models.efficientnet_b3(weights=None),
        'features': lambda m: nn.Sequential(m.features, m.avgpool),
        'feat_dim': 1536,
    },
    'efficientnet_v2_s': {
        'factory': lambda: models.efficientnet_v2_s(weights=None),
        'features': lambda m: nn.Sequential(m.features, m.avgpool),
        'feat_dim': 1280,
    },
    'resnet50': {
        'factory': lambda: models.resnet50(weights=None),
        'features': lambda m: nn.Sequential(*list(m.children())[:-1]),
        'feat_dim': 2048,
    },
}

def build_model(backbone: str, feat_dim: int, checkpoint: Path) -> nn.Module:
    cfg = _BACKBONES[backbone]
    m = cfg['factory']()
    features = cfg['features'](m)
    head = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(feat_dim,512), nn.ReLU(inplace=True),
        nn.Dropout(0.2), nn.Linear(512,20),
    )
    model = nn.Sequential(features, nn.Flatten(1), head)
    state = torch.load(checkpoint, map_location=device)
    # 支持 MultiLabelClassifier.state_dict() 格式（含 features.* 和 classifier.*）
    if any(k.startswith('features.') for k in state):
        feat_state = {k[len('features.'):]: v for k,v in state.items() if k.startswith('features.')}
        head_state = {k[len('classifier.'):]: v for k,v in state.items() if k.startswith('classifier.')}
        model[0].load_state_dict(feat_state)
        model[2].load_state_dict(head_state)
    else:
        model.load_state_dict(state)
    model.eval().to(device)
    return model

@torch.no_grad()
def infer_probs(model, df, split, img_size):
    ds = VOCDataset(df, DATA_DIR, split, val_tfm(img_size))
    dl = DataLoader(ds, EVAL_BS, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=pin, persistent_workers=NUM_WORKERS>0)
    probs = []
    for imgs, _ in tqdm(dl, leave=False, desc=f'{split} infer'):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
            p = (torch.sigmoid(model(imgs)) + torch.sigmoid(model(imgs.flip(-1)))) / 2
        probs.append(p.float().cpu().numpy())
    return np.vstack(probs), list(ds.indices)

In [ ]:
# ── Step 1：收集各模型的验证集概率 ────────────────────────────────────────────
full_df = pd.read_csv(DATA_DIR/'train'/'train_set.csv', index_col='Id')
_, vl_idx = train_test_split(range(len(full_df)), test_size=VAL_SPLIT, random_state=SEED)
val_df = full_df.iloc[vl_idx]
val_labels = val_df[LABELS].to_numpy(dtype=float)

exp_names   = []
val_prob_list  = []   # list of (N_val, 20) arrays
test_prob_list = []   # list of (750, 20) arrays

test_df = pd.read_csv(DATA_DIR/'test'/'test_set.csv', index_col='Id')

# 方案 C：从 checkpoint 推理
for exp_name, (backbone, feat_dim, ckpt_path, img_size) in CHECKPOINT_SOURCES.items():
    if not ckpt_path.exists():
        print(f'[skip] {exp_name}: checkpoint not found at {ckpt_path}')
        continue
    print(f'Loading {exp_name} from {ckpt_path}...')
    mdl = build_model(backbone, feat_dim, ckpt_path)
    vp, _ = infer_probs(mdl, val_df,  'train', img_size)
    tp, _ = infer_probs(mdl, test_df, 'test',  img_size)
    val_prob_list.append(vp)
    test_prob_list.append(tp)
    exp_names.append(exp_name)
    del mdl
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# 方案 B：从 CSV 加载
for exp_name, (val_csv, test_csv) in PROB_SOURCES.items():
    if not Path(val_csv).exists() or not Path(test_csv).exists():
        print(f'[skip] {exp_name}: CSV not found')
        continue
    print(f'Loading {exp_name} from CSV...')
    vp = pd.read_csv(val_csv, index_col='Id')[LABELS].to_numpy(dtype=float)
    tp = pd.read_csv(test_csv, index_col='Id')[LABELS].to_numpy(dtype=float)
    val_prob_list.append(vp)
    test_prob_list.append(tp)
    exp_names.append(exp_name)

if not exp_names:
    raise RuntimeError('没有可用的模型概率。请检查 CHECKPOINT_SOURCES 或 PROB_SOURCES 配置。')

print(f'\n已加载 {len(exp_names)} 个模型:', exp_names)
val_stack  = np.stack(val_prob_list,  axis=0)   # (M, N_val, 20)
test_stack = np.stack(test_prob_list, axis=0)   # (M, 750,   20)

In [ ]:
# ── Step 2：单模型基线（验证集）──────────────────────────────────────────────
baseline_rows = []
for i, name in enumerate(exp_names):
    p = val_prob_list[i]
    mAP = average_precision_score(val_labels, p, average='macro')
    # per-class 最优阈值 F1
    f1s = []
    for j in range(len(LABELS)):
        best_f1 = max(
            f1_score(val_labels[:,j], (p[:,j]>t).astype(int), zero_division=0)
            for t in np.arange(0.1, 0.91, 0.05)
        )
        f1s.append(best_f1)
    weak_f1 = np.mean([f1s[LABELS.index(c)] for c in WEAK_CLASSES])
    baseline_rows.append(dict(experiment=name, mAP=round(mAP,4), weak_class_f1=round(weak_f1,4)))
    print(f'{name:<30s} mAP={mAP:.4f}  weak_F1={weak_f1:.4f}')

pd.DataFrame(baseline_rows).to_csv(MET/'ensemble_baselines.csv', index=False)

In [ ]:
# ── Step 3：搜索最优权重（全局 + per-class）────────────────────────────────────
M = len(exp_names)
weight_values = np.arange(0.0, 1.0 + GRID_STEP/2, GRID_STEP)
candidate_weights = [
    w / w.sum() for w in
    (np.array(combo, dtype=float) for combo in product(weight_values, repeat=M))
    if sum(combo) > 0
]
print(f'权重候选数: {len(candidate_weights)}，搜索中...')

# 全局权重：所有类使用相同一组权重
best_global_w, best_global_mAP = None, -1.0
for w in tqdm(candidate_weights, desc='global search'):
    p = np.tensordot(w, val_stack, axes=(0,0))   # (N_val, 20)
    mAP = average_precision_score(val_labels, p, average='macro')
    if mAP > best_global_mAP: best_global_mAP, best_global_w = mAP, w.copy()

global_val_probs = np.tensordot(best_global_w, val_stack, axes=(0,0))
print(f'全局最优 mAP: {best_global_mAP:.4f}  weights: {[round(x,3) for x in best_global_w]}')

# Per-class 权重：每个类独立搜索
class_weights = np.zeros((len(LABELS), M), dtype=float)
class_ap_list = []
for ci, cls in enumerate(tqdm(LABELS, desc='per-class search')):
    best_ap, best_cw = -1.0, None
    for w in candidate_weights:
        p_cls = np.tensordot(w, val_stack[:,:,ci], axes=(0,0))   # (N_val,)
        ap = average_precision_score(val_labels[:,ci], p_cls)
        if ap > best_ap: best_ap, best_cw = ap, w.copy()
    class_weights[ci] = best_cw
    class_ap_list.append(best_ap)

def apply_per_class(stack, cw):
    out = np.zeros(stack.shape[1:], dtype=float)
    for ci in range(len(LABELS)):
        out[:,ci] = np.tensordot(cw[ci], stack[:,:,ci], axes=(0,0))
    return out

per_class_val_probs = apply_per_class(val_stack, class_weights)
per_class_mAP = float(np.mean(class_ap_list))
print(f'Per-class 加权 mAP: {per_class_mAP:.4f}')

In [ ]:
# ── Step 4：选择更优的融合方式，确定最终验证集概率和阈值 ───────────────────────
def find_thresholds(probs, labels):
    thresholds = np.zeros(len(LABELS))
    for i in range(len(LABELS)):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.91, 0.05):
            preds = (probs[:,i] > t).astype(int)
            tp = (preds * labels[:,i]).sum()
            f1 = 2*tp / (2*tp + (preds*(1-labels[:,i])).sum() + ((1-preds)*labels[:,i]).sum() + 1e-8)
            if f1 > best_f1: best_f1, best_t = f1, t
        thresholds[i] = best_t
    return thresholds

def mean_f1(probs, labels, thresholds):
    preds = (probs > thresholds[None,:]).astype(int)
    return np.mean([f1_score(labels[:,i], preds[:,i], zero_division=0) for i in range(len(LABELS))])

t_global    = find_thresholds(global_val_probs,    val_labels)
t_per_class = find_thresholds(per_class_val_probs, val_labels)
f1_global    = mean_f1(global_val_probs,    val_labels, t_global)
f1_per_class = mean_f1(per_class_val_probs, val_labels, t_per_class)
print(f'全局权重   mean-F1={f1_global:.4f}')
print(f'Per-class  mean-F1={f1_per_class:.4f}')

if f1_per_class >= f1_global:
    chosen_mode = 'per_class'
    final_val_probs = per_class_val_probs
    thresholds = t_per_class
else:
    chosen_mode = 'global'
    final_val_probs = global_val_probs
    thresholds = t_global
print(f'\n选定模式: {chosen_mode}')

np.save(MET/'best_thresholds.npy', thresholds)

# 保存权重 JSON
weights_payload = {
    'experiments': exp_names,
    'chosen_mode': chosen_mode,
    'global_weights': best_global_w.tolist(),
    'global_mAP': best_global_mAP,
    'per_class_mAP': per_class_mAP,
    'thresholds': thresholds.tolist(),
    'per_class_weights': {label: class_weights[i].tolist() for i, label in enumerate(LABELS)},
}
(MET/'ensemble_weights.json').write_text(json.dumps(weights_payload, indent=2), encoding='utf-8')
print('Weights saved:', MET/'ensemble_weights.json')

In [ ]:
# ── Step 5：可视化各模型对比 + ensemble 结果 ──────────────────────────────────
# Per-class AP 对比图
ap_rows = []
for i, name in enumerate(exp_names):
    for j, cls in enumerate(LABELS):
        ap_rows.append(dict(model=name, cls=cls, ap=average_precision_score(val_labels[:,j], val_stack[i,:,j])))
ens_ap = [average_precision_score(val_labels[:,j], final_val_probs[:,j]) for j in range(len(LABELS))]
for j, cls in enumerate(LABELS):
    ap_rows.append(dict(model='ensemble', cls=cls, ap=ens_ap[j]))

ap_df = pd.DataFrame(ap_rows)
ap_df.to_csv(MET/'ap_per_class_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(14,5))
x = np.arange(len(LABELS))
bar_w = 0.8 / (len(exp_names)+1)
colors = ['#4c86b7','#5ba85a','#e07b39','#9b59b6','#e74c3c']
all_models = exp_names + ['ensemble']
for mi, mname in enumerate(all_models):
    aps = ap_df[ap_df['model']==mname].set_index('cls').loc[LABELS,'ap'].values
    offset = (mi - len(all_models)/2 + 0.5) * bar_w
    ax.bar(x+offset, aps, bar_w, label=mname, color=colors[mi % len(colors)], alpha=0.85)

ens_mAP = float(np.mean(ens_ap))
ax.axhline(ens_mAP, color='black', linestyle='--', lw=1.2, label=f'Ensemble mAP={ens_mAP:.3f}')
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=45, ha='right')
ax.set_ylim(0,1.1); ax.set_ylabel('AP'); ax.legend(loc='lower right', fontsize=8)
ax.set_title('Per-class AP: 各模型 vs Ensemble')
fig.tight_layout()
fig.savefig(FIG/'ensemble_ap_comparison.png', dpi=140); plt.show()
print(f'Ensemble val mAP: {ens_mAP:.6f}')

In [ ]:
# ── Step 6：生成测试集 ensemble 概率 + 提交 CSV ───────────────────────────────
if chosen_mode == 'per_class':
    final_test_probs = apply_per_class(test_stack, class_weights)
else:
    final_test_probs = np.tensordot(best_global_w, test_stack, axes=(0,0))

test_ids = list(test_df.index)
preds = (final_test_probs > thresholds[None,:]).astype(int)

prob_df = pd.DataFrame(final_test_probs, columns=LABELS, index=test_ids); prob_df.index.name='Id'
pred_df = pd.DataFrame(preds,            columns=LABELS, index=test_ids); pred_df.index.name='Id'
prob_df.to_csv(PRED/f'test_probabilities_{ENSEMBLE_NAME}.csv')
pred_df.to_csv(PRED/f'test_binary_predictions_{ENSEMBLE_NAME}.csv')

rows = {'Id':[], 'Predicted':[]}
for idx in pred_df.index:
    rows['Id'].append(f'{idx}_classification')
    rows['Predicted'].append(rle_encode(pred_df.loc[idx, LABELS].values.astype(int)))
    rows['Id'].append(f'{idx}_segmentation')
    rows['Predicted'].append('')
sub = pd.DataFrame(rows).set_index('Id')
sub_path = SUB / f'submission_classification_{ENSEMBLE_NAME}.csv'
sub.to_csv(sub_path)
print(f'提交文件: {sub_path} ({len(sub)} 行)')
sub.head(4)

In [ ]:
# ── 汇总报告 ───────────────────────────────────────────────────────────────────
print('='*60)
print(f'Ensemble 模式  : {chosen_mode}')
print(f'包含模型       : {exp_names}')
print(f'全局权重 mAP   : {best_global_mAP:.4f}')
print(f'Per-class mAP  : {per_class_mAP:.4f}')
print(f'选定 mAP       : {ens_mAP:.4f}')
print(f'Weak-class F1  : {np.mean([f1_score(val_labels[:,LABELS.index(c)], (final_val_probs[:,LABELS.index(c)]>thresholds[LABELS.index(c)]).astype(int), zero_division=0) for c in WEAK_CLASSES]):.4f}')
print(f'提交文件行数   : {len(sub)}')
print(f'输出目录       : {OUT}')
print('='*60)
print('\n各模型基线 mAP:')
for r in baseline_rows:
    print(f"  {r['experiment']:<30s} mAP={r['mAP']}")